In [1]:
# ============================================================
# DAY 9 — ML MODELS ON FULL UNIVERSE (751 STOCKS)
# AI Stock Screener — Indian Markets
# Models: Reversal Classifier, Trend Classifier, Price Forecast
# Report: Short + Long format (matching Day 8 weekly report)
# ============================================================

# ── CELL 1: IMPORTS & SETUP ──────────────────────────────────
import pandas as pd
import numpy as np
import pickle
import warnings
import os
warnings.filterwarnings('ignore')

from xgboost import XGBClassifier, XGBRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score,
                             classification_report)
from collections import defaultdict
from datetime import date, datetime

os.makedirs('../models', exist_ok=True)
os.makedirs('../data/weekly_reports', exist_ok=True)

print("✅ Imports done")

✅ Imports done


In [2]:
# ── CELL 2: LOAD ALL DATA ────────────────────────────────────
# Full universe pkl files (751 stocks)

with open('../data/price_data_full.pkl', 'rb') as f:
    price_data = pickle.load(f)

with open('../data/indicator_data_full.pkl', 'rb') as f:
    indicator_data = pickle.load(f)

fund_scores = pd.read_csv('../data/fundamental_scores_full.csv')
tech_report = pd.read_csv('../data/technical_report_full.csv')
prefilt_df  = pd.read_csv('../data/prefilt_passed.csv')

# Market Cap comes from prefilt — merge if not already in fund_scores
if 'Market_Cap_Cr' not in fund_scores.columns:
    fund_scores = fund_scores.merge(
        prefilt_df[['Symbol', 'Market_Cap_Cr']], on='Symbol', how='left'
    )

# Build lookup: symbol → {Sector, Cap Category, Market Cap Cr, Final Score}
fund_lookup = tech_report.set_index('Symbol')[
    ['Sector', 'Cap Category', 'Market Cap Cr']
].to_dict('index')

for sym, row in fund_scores.set_index('Symbol').iterrows():
    if sym in fund_lookup:
        fund_lookup[sym]['Final Score'] = row.get('Final Score', 50)

print(f"✅ Price data     : {len(price_data)} stocks")
print(f"✅ Indicator data : {len(indicator_data)} stocks")
print(f"✅ Fund scores    : {len(fund_scores)} stocks")
print(f"✅ Tech report    : {len(tech_report)} stocks")
print(f"✅ Lookup built   : {len(fund_lookup)} stocks")

print(f"\nSectors in universe:")
print(tech_report['Sector'].value_counts().to_string())

print(f"\nCap Categories:")
print(tech_report['Cap Category'].value_counts().to_string())

✅ Price data     : 752 stocks
✅ Indicator data : 752 stocks
✅ Fund scores    : 752 stocks
✅ Tech report    : 752 stocks
✅ Lookup built   : 752 stocks

Sectors in universe:
Sector
Capital Goods                         130
Financial Services                     91
Healthcare                             68
Information Technology                 60
Automobile and Auto Components         59
Chemicals                              57
Fast Moving Consumer Goods             54
Consumer Durables                      40
Consumer Services                      26
Construction                           25
Textiles                               25
Services                               24
Metals & Mining                        23
Oil, Gas & Consumable Fuels            20
Power                                  12
Realty                                  9
Construction Materials                  6
Utilities                               6
Telecommunication                       6
Media, Entertainment & 

In [3]:
# ── CELL 3: GROUP ASSIGNMENT ─────────────────────────────────
# 6 groups — all 22 sectors from the full universe mapped explicitly

SECTOR_GROUP_MAP = {
    # IT
    'Information Technology'              : 'IT',
    # Financial
    'Financial Services'                  : 'Financial',
    # Chemicals
    'Chemicals'                           : 'Chemicals',
    # Healthcare
    'Healthcare'                          : 'Healthcare',
    # Consumer
    'Consumer Durables'                   : 'Consumer',
    'Consumer Services'                   : 'Consumer',
    'Fast Moving Consumer Goods'          : 'Consumer',
    # Industrial — everything else
    'Capital Goods'                       : 'Industrial',
    'Automobile and Auto Components'      : 'Industrial',
    'Construction'                        : 'Industrial',
    'Construction Materials'              : 'Industrial',
    'Textiles'                            : 'Industrial',
    'Services'                            : 'Industrial',
    'Metals & Mining'                     : 'Industrial',
    'Oil, Gas & Consumable Fuels'         : 'Industrial',
    'Power'                               : 'Industrial',
    'Realty'                              : 'Industrial',
    'Utilities'                           : 'Industrial',
    'Telecommunication'                   : 'Industrial',
    'Media, Entertainment & Publication'  : 'Industrial',
    'Forest Materials'                    : 'Industrial',
    'Diversified'                         : 'Industrial',
}

symbol_group = {}
for sym, info in fund_lookup.items():
    sector = info.get('Sector', '')
    grp    = SECTOR_GROUP_MAP.get(sector)
    if grp is None:
        print(f"  ⚠️  Unmapped sector: '{sector}' ({sym}) → Industrial")
        grp = 'Industrial'
    symbol_group[sym] = grp

group_stocks = defaultdict(list)
for sym, grp in symbol_group.items():
    group_stocks[grp].append(sym)

print("Group assignments:")
for grp, stocks in sorted(group_stocks.items()):
    print(f"  {grp:12} : {len(stocks):3d} stocks")

print(f"\nTotal mapped : {len(symbol_group)} stocks")
print(f"Total groups : {len(group_stocks)}")

Group assignments:
  Chemicals    :  57 stocks
  Consumer     : 120 stocks
  Financial    :  91 stocks
  Healthcare   :  68 stocks
  IT           :  60 stocks
  Industrial   : 356 stocks

Total mapped : 752 stocks
Total groups : 6


In [4]:
# ── CELL 4: FEATURE ENGINEERING ──────────────────────────────
def build_features(symbol, price_df, ind_df):
    df = price_df.copy()

    # Drop overlapping columns before joining
    overlap_cols = ['Open', 'High', 'Low', 'Close', 'Volume',
                    'Dividends', 'Stock Splits']
    ind_clean = ind_df.drop(
        columns=[c for c in overlap_cols if c in ind_df.columns]
    )
    df = df.join(ind_clean, how='left')

    # ── PRICE FEATURES ───────────────────────────────────────
    df['return_1d']  = df['Close'].pct_change(1)
    df['return_5d']  = df['Close'].pct_change(5)
    df['return_20d'] = df['Close'].pct_change(20)
    df['return_60d'] = df['Close'].pct_change(60)

    df['high_52w']      = df['High'].rolling(252).max()
    df['low_52w']       = df['Low'].rolling(252).min()
    df['dist_52w_high'] = (df['Close'] - df['high_52w']) / df['high_52w']
    df['dist_52w_low']  = (df['Close'] - df['low_52w'])  / df['low_52w']

    df['tr'] = np.maximum(
        df['High'] - df['Low'],
        np.maximum(
            abs(df['High'] - df['Close'].shift(1)),
            abs(df['Low']  - df['Close'].shift(1))
        )
    )
    df['atr_14']         = df['tr'].rolling(14).mean()
    df['atr_pct']        = df['atr_14'] / df['Close']
    df['volatility_20d'] = df['return_1d'].rolling(20).std()
    df['volatility_60d'] = df['return_1d'].rolling(60).std()

    # ── VOLUME FEATURES ──────────────────────────────────────
    df['vol_ma20']      = df['Volume'].rolling(20).mean()
    df['vol_ratio_5d']  = df['Volume'].rolling(5).mean()  / df['vol_ma20']
    df['vol_ratio_20d'] = df['Volume'].rolling(20).mean() / df['Volume'].rolling(60).mean()
    df['obv']           = (np.sign(df['Close'].diff()) * df['Volume']).fillna(0).cumsum()
    df['obv_slope_10d'] = df['obv'].diff(10) / (df['obv'].abs().rolling(10).mean() + 1e-9)
    df['vol_spike']     = (df['Volume'] > df['vol_ma20'] * 2).astype(int)

    # ── RSI FEATURES ─────────────────────────────────────────
    if 'RSI' in df.columns:
        df['rsi']            = df['RSI']
        df['rsi_slope_5d']   = df['RSI'].diff(5)
        df['rsi_oversold']   = (df['RSI'] < 30).astype(int)
        df['rsi_overbought'] = (df['RSI'] > 70).astype(int)

    # ── MACD FEATURES ────────────────────────────────────────
    if 'MACD_Hist' in df.columns:
        df['macd_hist']     = df['MACD_Hist']
        df['macd_slope_3d'] = df['MACD_Hist'].diff(3)
        df['macd_slope_5d'] = df['MACD_Hist'].diff(5)
        df['macd_cross']    = (
            (df['MACD_Hist'] > 0) & (df['MACD_Hist'].shift(1) <= 0)
        ).astype(int)

    # ── ADX FEATURES ─────────────────────────────────────────
    if 'ADX' in df.columns:
        df['adx']       = df['ADX']
        df['adx_slope'] = df['ADX'].diff(5)
        # Day 8 indicator pkl uses DI_Plus / DI_Minus
        # Cell 5 post-processing will fix di_spread for all stocks
        if 'DI+' in df.columns:
            df['di_spread'] = df['DI+'] - df['DI-']
        elif 'DI_Plus' in df.columns:
            df['di_spread'] = df['DI_Plus'] - df['DI_Minus']
        else:
            df['di_spread'] = 0

    # ── EMA FEATURES ─────────────────────────────────────────
    if 'EMA20' in df.columns:
        df['price_vs_ema20']  = (df['Close'] - df['EMA20'])  / df['EMA20']
        df['price_vs_ema50']  = (df['Close'] - df['EMA50'])  / df['EMA50']
        df['price_vs_ema200'] = (df['Close'] - df['EMA200']) / df['EMA200']
        df['ema20_vs_ema50']  = (df['EMA20'] - df['EMA50'])  / df['EMA50']
        df['ema50_vs_ema200'] = (df['EMA50'] - df['EMA200']) / df['EMA200']

    # ── TARGET VARIABLES ─────────────────────────────────────
    df['future_return_20d'] = df['Close'].shift(-20) / df['Close'] - 1
    df['future_return_5d']  = df['Close'].shift(-5)  / df['Close'] - 1

    df['bottom_reversal'] = (
        (df['return_20d'] < -0.05) &
        (df['future_return_20d'] > 0.08)
    ).astype(int)

    df['top_reversal'] = (
        (df['return_20d'] > 0.05) &
        (df['future_return_20d'] < -0.08)
    ).astype(int)

    # trend_label placeholder — rebuilt properly in Cell 6
    df['trend_label'] = 'Sideways'

    df['target_25d']  = df['Close'].shift(-25)  / df['Close'] - 1
    df['target_45d']  = df['Close'].shift(-45)  / df['Close'] - 1
    df['target_180d'] = df['Close'].shift(-180) / df['Close'] - 1

    return df

# ── BUILD FEATURES FOR ALL STOCKS ────────────────────────────
print("Building features for all stocks...")
all_features = {}

for symbol in price_data.keys():
    price_df = price_data[symbol]
    ind_df   = indicator_data.get(symbol, pd.DataFrame())
    try:
        feat_df             = build_features(symbol, price_df, ind_df)
        all_features[symbol] = feat_df
    except Exception as e:
        print(f"  ⚠️  {symbol}: {e}")

print(f"✅ Features built for {len(all_features)} stocks")

sample    = list(all_features.keys())[0]
sample_df = all_features[sample].dropna(subset=['return_20d'])
print(f"\nSample stock  : {sample}")
print(f"Rows (all)    : {len(all_features[sample])}")
print(f"Rows (usable) : {len(sample_df)}")
print(f"Columns total : {len(sample_df.columns)}")

Building features for all stocks...
✅ Features built for 752 stocks

Sample stock  : 20MICRONS
Rows (all)    : 4304
Rows (usable) : 4284
Columns total : 60


In [5]:
# ── CELL 5: DI_SPREAD FIX + DEFINE FEATURE COLUMNS ───────────

# Fix di_spread for all stocks using Day 8 column names (DI_Plus / DI_Minus)
for symbol in all_features:
    df = all_features[symbol]
    if 'DI_Plus' in df.columns and 'DI_Minus' in df.columns:
        df['di_spread'] = df['DI_Plus'] - df['DI_Minus']
    elif 'DI+' in df.columns and 'DI-' in df.columns:
        df['di_spread'] = df['DI+'] - df['DI-']
    if 'di_spread' not in df.columns:
        df['di_spread'] = 0
    all_features[symbol] = df

# Input features — identical to Day 7 (30 features)
FEATURE_COLS = [
    # Price
    'return_1d', 'return_5d', 'return_20d', 'return_60d',
    'dist_52w_high', 'dist_52w_low', 'atr_pct',
    'volatility_20d', 'volatility_60d',
    # Volume
    'vol_ratio_5d', 'vol_ratio_20d', 'obv_slope_10d', 'vol_spike',
    # RSI
    'rsi', 'rsi_slope_5d', 'rsi_oversold', 'rsi_overbought',
    # MACD
    'macd_hist', 'macd_slope_3d', 'macd_slope_5d', 'macd_cross',
    # ADX
    'adx', 'adx_slope', 'di_spread',
    # EMA
    'price_vs_ema20', 'price_vs_ema50', 'price_vs_ema200',
    'ema20_vs_ema50', 'ema50_vs_ema200',
]

TARGET_BOTTOM = 'bottom_reversal'
TARGET_TOP    = 'top_reversal'
TARGET_TREND  = 'trend_label'
TARGET_25D    = 'target_25d'
TARGET_45D    = 'target_45d'
TARGET_180D   = 'target_180d'

# Verify di_spread is present and non-zero for a sample stock
sample = list(all_features.keys())[0]
df_s   = all_features[sample]
print(f"di_spread check ({sample}):")
print(f"  Column present : {'di_spread' in df_s.columns}")
print(f"  Non-zero rows  : {(df_s['di_spread'] != 0).sum()}")
print(f"  Sample values  : {df_s['di_spread'].dropna().tail(3).values}")
print(f"\n✅ FEATURE_COLS defined: {len(FEATURE_COLS)} features")

di_spread check (20MICRONS):
  Column present : True
  Non-zero rows  : 4303
  Sample values  : [-19.74775572 -25.56620665 -23.42302041]

✅ FEATURE_COLS defined: 29 features


In [24]:
# ── CELL 6: REBUILD TREND LABELS + CONTINUATION LABELS ───────
# trend_label      → used for Model 2 training (3-class)
# bottom_reversal  → already defined in build_features()
# top_reversal     → already defined in build_features()
# bullish_cont     → NEW: stock already in uptrend + continues up
# bearish_cont     → NEW: stock already in downtrend + continues down

# ── TREND LABEL (3-class, no leakage) ────────────────────────
def assign_trend_3class(row):
    f20 = row.get('future_return_20d', 0)
    if pd.isna(f20):
        return 'Sideways'
    if f20 > 0.05:
        return 'Uptrend'
    elif f20 < -0.05:
        return 'Downtrend'
    else:
        return 'Sideways'

for symbol in all_features:
    df = all_features[symbol]
    df['trend_label'] = df.apply(assign_trend_3class, axis=1)
    all_features[symbol] = df

# ── CONTINUATION LABELS (NEW) ─────────────────────────────────
# Bullish Continuation:
#   Current context  → already in uptrend (price > EMA200, EMA20 > EMA50,
#                       return_60d > 5%)
#   Future context   → continues up (future_return_20d > 3%)
#   Logic: we use a lower threshold (3% not 5%) because continuation
#   in an established trend doesn't need a big move to be meaningful

# Bearish Continuation:
#   Current context  → already in downtrend (price < EMA200, EMA20 < EMA50,
#                       return_60d < -5%)
#   Future context   → continues down (future_return_20d < -3%)

def assign_continuation_labels(df):
    # Current trend context — no future data used here
    price_above_ema200 = df['Close'] > df['EMA200']
    ema_bullish_align  = (df['EMA20'] > df['EMA50']) & (df['EMA50'] > df['EMA200'])
    ema_bearish_align  = (df['EMA20'] < df['EMA50']) & (df['EMA50'] < df['EMA200'])
    strong_uptrend_60d = df['return_60d'] > 0.05
    strong_dntrend_60d = df['return_60d'] < -0.05

    # Future return — this is the target (shift already applied in build_features)
    future_up   = df['future_return_20d'] > 0.03
    future_down = df['future_return_20d'] < -0.03

    # Bullish continuation: in uptrend context AND continues up
    df['bullish_cont'] = (
        price_above_ema200 &
        ema_bullish_align  &
        strong_uptrend_60d &
        future_up
    ).astype(int)

    # Bearish continuation: in downtrend context AND continues down
    df['bearish_cont'] = (
        ~price_above_ema200 &
        ema_bearish_align   &
        strong_dntrend_60d  &
        future_down
    ).astype(int)

    return df

print("Building continuation labels for all stocks...")
missing = 0
for symbol in all_features:
    df = all_features[symbol]
    required = ['Close', 'EMA200', 'EMA20', 'EMA50',
                'return_60d', 'future_return_20d']
    if all(c in df.columns for c in required):
        all_features[symbol] = assign_continuation_labels(df)
    else:
        # If EMA columns missing, set both to 0
        df['bullish_cont'] = 0
        df['bearish_cont'] = 0
        all_features[symbol] = df
        missing += 1

if missing > 0:
    print(f"  ⚠️  {missing} stocks missing EMA columns — set to 0")

# ── VERIFY DISTRIBUTIONS ──────────────────────────────────────
sample = list(all_features.keys())[0]
df_s   = all_features[sample]

print(f"\nSample stock: {sample}")
print(f"\nTrend label distribution:")
print(df_s['trend_label'].value_counts().to_string())

print(f"\nBullish continuation events : {df_s['bullish_cont'].sum()}")
print(f"Bearish continuation events : {df_s['bearish_cont'].sum()}")
print(f"Total rows                  : {len(df_s)}")

# Check across full universe
total_bullish_cont = sum(
    all_features[s]['bullish_cont'].sum() for s in all_features
)
total_bearish_cont = sum(
    all_features[s]['bearish_cont'].sum() for s in all_features
)
total_rows = sum(len(all_features[s]) for s in all_features)

print(f"\nFull universe:")
print(f"  Bullish cont events : {total_bullish_cont:,} / {total_rows:,} rows "
      f"({total_bullish_cont/total_rows*100:.1f}%)")
print(f"  Bearish cont events : {total_bearish_cont:,} / {total_rows:,} rows "
      f"({total_bearish_cont/total_rows*100:.1f}%)")
print(f"\n✅ Continuation labels built for {len(all_features)} stocks")

Building continuation labels for all stocks...

Sample stock: 20MICRONS

Trend label distribution:
trend_label
Sideways     1610
Uptrend      1447
Downtrend    1247

Bullish continuation events : 626
Bearish continuation events : 244
Total rows                  : 4304

Full universe:
  Bullish cont events : 414,189 / 2,585,199 rows (16.0%)
  Bearish cont events : 193,762 / 2,585,199 rows (7.5%)

✅ Continuation labels built for 752 stocks


In [25]:
# ── CELL 7: WALK-FORWARD HELPERS ─────────────────────────────

WF_FOLDS = [
    ('2015-01-01', '2020-12-31', '2021-01-01', '2021-12-31'),
    ('2015-01-01', '2021-12-31', '2022-01-01', '2022-12-31'),
    ('2015-01-01', '2022-12-31', '2023-01-01', '2023-12-31'),
    ('2015-01-01', '2023-12-31', '2024-01-01', '2024-12-31'),
    ('2015-01-01', '2024-12-31', '2025-01-01', '2025-12-31'),
]

def prepare_group_data(group_name, target_col):
    """
    Combine all stocks in a group into one training DataFrame.
    Encodes symbol as integer feature via LabelEncoder.
    """
    group_syms = group_stocks[group_name]
    frames = []

    for sym in group_syms:
        if sym not in all_features:
            continue
        df = all_features[sym].copy()
        df['symbol_id'] = sym
        keep = FEATURE_COLS + [target_col, 'symbol_id']
        df   = df[keep].copy()
        df   = df.replace([np.inf, -np.inf], np.nan)
        df   = df.dropna()
        df.index = all_features[sym].dropna(
            subset=FEATURE_COLS + [target_col]
        ).index
        frames.append(df)

    if not frames:
        return None

    combined = pd.concat(frames).sort_index()
    le = LabelEncoder()
    combined['symbol_enc'] = le.fit_transform(combined['symbol_id'])
    return combined, le


def walk_forward_classify_v2(group_name, target_col):
    """
    Reversal classifier with:
    - scale_pos_weight to handle class imbalance
    - aucpr eval metric (better than logloss for rare events)
    - precision / recall / f1 reported per fold
    """
    result = prepare_group_data(group_name, target_col)
    if result is None:
        return None, [], None

    combined, le  = result
    feat_cols     = FEATURE_COLS + ['symbol_enc']
    fold_metrics  = []

    for tr_start, tr_end, te_start, te_end in WF_FOLDS:
        train_df = combined[tr_start:tr_end]
        test_df  = combined[te_start:te_end]

        if len(train_df) < 100 or len(test_df) < 10:
            continue
        if train_df[target_col].nunique() < 2:
            continue

        X_train = train_df[feat_cols]
        y_train = train_df[target_col]
        X_test  = test_df[feat_cols]
        y_test  = test_df[target_col]

        neg_count = (y_train == 0).sum()
        pos_count = (y_train == 1).sum()
        scale_pos = neg_count / pos_count if pos_count > 0 else 1

        model = XGBClassifier(
            n_estimators     = 200,
            max_depth        = 4,
            learning_rate    = 0.05,
            subsample        = 0.8,
            colsample_bytree = 0.8,
            scale_pos_weight = scale_pos,
            use_label_encoder= False,
            eval_metric      = 'aucpr',
            random_state     = 42,
        )
        model.fit(X_train, y_train,
                  eval_set=[(X_test, y_test)],
                  verbose=False)

        preds = model.predict(X_test)
        fold_metrics.append({
            'fold'      : te_start[:4],
            'accuracy'  : round(accuracy_score(y_test, preds), 3),
            'precision' : round(precision_score(y_test, preds, zero_division=0), 3),
            'recall'    : round(recall_score(y_test, preds, zero_division=0), 3),
            'f1'        : round(f1_score(y_test, preds, zero_division=0), 3),
            'test_rows' : len(y_test),
            'pos_events': int(y_test.sum()),
        })

    # Final model trained on ALL data
    neg_all   = (combined[target_col] == 0).sum()
    pos_all   = (combined[target_col] == 1).sum()
    scale_all = neg_all / pos_all if pos_all > 0 else 1

    final_model = XGBClassifier(
        n_estimators     = 200,
        max_depth        = 4,
        learning_rate    = 0.05,
        subsample        = 0.8,
        colsample_bytree = 0.8,
        scale_pos_weight = scale_all,
        use_label_encoder= False,
        eval_metric      = 'aucpr',
        random_state     = 42,
    )
    final_model.fit(combined[feat_cols], combined[target_col], verbose=False)
    return final_model, fold_metrics, le


def walk_forward_trend(group_name):
    """
    3-class trend classifier.
    Returns: model, fold_metrics, symbol_encoder, label_encoder
    Note: two separate encoders — le_sym for symbol_enc feature,
    le_trend for decoding predicted class back to label name.
    """
    result = prepare_group_data(group_name, TARGET_TREND)
    if result is None:
        return None, [], None, None

    combined, le_sym  = result
    feat_cols         = FEATURE_COLS + ['symbol_enc']

    le_trend = LabelEncoder()
    combined['trend_enc'] = le_trend.fit_transform(combined[TARGET_TREND])

    fold_metrics = []

    for tr_start, tr_end, te_start, te_end in WF_FOLDS:
        train_df = combined[tr_start:tr_end]
        test_df  = combined[te_start:te_end]

        if len(train_df) < 100 or len(test_df) < 10:
            continue
        if train_df['trend_enc'].nunique() < 2:
            continue

        X_train = train_df[feat_cols]
        y_train = train_df['trend_enc']
        X_test  = test_df[feat_cols]
        y_test  = test_df['trend_enc']

        model = XGBClassifier(
            n_estimators     = 300,
            max_depth        = 5,
            learning_rate    = 0.05,
            subsample        = 0.8,
            colsample_bytree = 0.8,
            use_label_encoder= False,
            eval_metric      = 'mlogloss',
            random_state     = 42,
        )
        model.fit(X_train, y_train,
                  eval_set=[(X_test, y_test)],
                  verbose=False)

        preds = model.predict(X_test)
        acc   = accuracy_score(y_test, preds)
        fold_metrics.append({
            'fold'     : te_start[:4],
            'accuracy' : round(acc, 3),
            'test_rows': len(y_test),
        })

    # Final model on ALL data
    final_model = XGBClassifier(
        n_estimators     = 300,
        max_depth        = 5,
        learning_rate    = 0.05,
        subsample        = 0.8,
        colsample_bytree = 0.8,
        use_label_encoder= False,
        eval_metric      = 'mlogloss',
        random_state     = 42,
    )
    final_model.fit(combined[feat_cols], combined['trend_enc'], verbose=False)
    return final_model, fold_metrics, le_sym, le_trend


# ── ADD TO CELL 7: CONTINUATION CLASSIFIER HELPER ────────────
# Add this function at the end of Cell 7
# All existing helpers (walk_forward_classify_v2, walk_forward_trend,
# walk_forward_regress) remain unchanged

def walk_forward_continuation(group_name, target_col):
    """
    Continuation classifier:
    - Bullish cont: stock in uptrend + predicted to continue up
    - Bearish cont: stock in downtrend + predicted to continue down
    - Binary classifier like reversal but with different label definition
    - scale_pos_weight used since continuation events are ~10-16% of rows
    - eval_metric: aucpr (same as reversal — good for imbalanced classes)
    - Reports precision / recall / f1 per fold
    """
    result = prepare_group_data(group_name, target_col)
    if result is None:
        return None, [], None

    combined, le  = result
    feat_cols     = FEATURE_COLS + ['symbol_enc']
    fold_metrics  = []

    for tr_start, tr_end, te_start, te_end in WF_FOLDS:
        train_df = combined[tr_start:tr_end]
        test_df  = combined[te_start:te_end]

        if len(train_df) < 100 or len(test_df) < 10:
            continue
        if train_df[target_col].nunique() < 2:
            continue

        X_train = train_df[feat_cols]
        y_train = train_df[target_col]
        X_test  = test_df[feat_cols]
        y_test  = test_df[target_col]

        neg_count = (y_train == 0).sum()
        pos_count = (y_train == 1).sum()
        scale_pos = neg_count / pos_count if pos_count > 0 else 1

        model = XGBClassifier(
            n_estimators     = 200,
            max_depth        = 4,
            learning_rate    = 0.05,
            subsample        = 0.8,
            colsample_bytree = 0.8,
            scale_pos_weight = scale_pos,
            use_label_encoder= False,
            eval_metric      = 'aucpr',
            random_state     = 42,
        )
        model.fit(X_train, y_train,
                  eval_set=[(X_test, y_test)],
                  verbose=False)

        preds = model.predict(X_test)
        fold_metrics.append({
            'fold'      : te_start[:4],
            'accuracy'  : round(accuracy_score(y_test, preds), 3),
            'precision' : round(precision_score(y_test, preds, zero_division=0), 3),
            'recall'    : round(recall_score(y_test, preds, zero_division=0), 3),
            'f1'        : round(f1_score(y_test, preds, zero_division=0), 3),
            'test_rows' : len(y_test),
            'pos_events': int(y_test.sum()),
        })

    # Final model on ALL data
    neg_all   = (combined[target_col] == 0).sum()
    pos_all   = (combined[target_col] == 1).sum()
    scale_all = neg_all / pos_all if pos_all > 0 else 1

    final_model = XGBClassifier(
        n_estimators     = 200,
        max_depth        = 4,
        learning_rate    = 0.05,
        subsample        = 0.8,
        colsample_bytree = 0.8,
        scale_pos_weight = scale_all,
        use_label_encoder= False,
        eval_metric      = 'aucpr',
        random_state     = 42,
    )
    final_model.fit(combined[feat_cols], combined[target_col], verbose=False)
    return final_model, fold_metrics, le

print("✅ Continuation classifier helper added")
print(f"   Folds  : {len(WF_FOLDS)} (2021 → 2025)")
print(f"   Groups : {sorted(group_stocks.keys())}")


def walk_forward_regress(group_name, target_col):
    """
    Forecast regressor.
    Metric: directional accuracy (did we get the sign right?)
    """
    result = prepare_group_data(group_name, target_col)
    if result is None:
        return None, [], None

    combined, le  = result
    feat_cols     = FEATURE_COLS + ['symbol_enc']
    fold_metrics  = []

    for tr_start, tr_end, te_start, te_end in WF_FOLDS:
        train_df = combined[tr_start:tr_end]
        test_df  = combined[te_start:te_end]

        if len(train_df) < 100 or len(test_df) < 10:
            continue

        X_train = train_df[feat_cols]
        y_train = train_df[target_col]
        X_test  = test_df[feat_cols]
        y_test  = test_df[target_col]

        model = XGBRegressor(
            n_estimators     = 200,
            max_depth        = 4,
            learning_rate    = 0.05,
            subsample        = 0.8,
            colsample_bytree = 0.8,
            random_state     = 42,
        )
        model.fit(X_train, y_train,
                  eval_set=[(X_test, y_test)],
                  verbose=False)

        preds   = model.predict(X_test)
        dir_acc = np.mean(np.sign(preds) == np.sign(y_test))
        mae     = np.mean(np.abs(preds - y_test))
        fold_metrics.append({
            'fold'     : te_start[:4],
            'dir_acc'  : round(dir_acc, 3),
            'mae'      : round(mae, 3),
            'test_rows': len(y_test),
        })

    # Final model on ALL data
    final_model = XGBRegressor(
        n_estimators     = 200,
        max_depth        = 4,
        learning_rate    = 0.05,
        subsample        = 0.8,
        colsample_bytree = 0.8,
        random_state     = 42,
    )
    final_model.fit(combined[feat_cols], combined[target_col], verbose=False)
    return final_model, fold_metrics, le

print("✅ Walk-forward helpers defined")
print(f"   Folds  : {len(WF_FOLDS)} (2021 → 2025)")
print(f"   Groups : {sorted(group_stocks.keys())}")

✅ Continuation classifier helper added
   Folds  : 5 (2021 → 2025)
   Groups : ['Chemicals', 'Consumer', 'Financial', 'Healthcare', 'IT', 'Industrial']
✅ Walk-forward helpers defined
   Folds  : 5 (2021 → 2025)
   Groups : ['Chemicals', 'Consumer', 'Financial', 'Healthcare', 'IT', 'Industrial']


In [18]:
# ── CELL 8: TRAIN MODEL 1 — REVERSAL CLASSIFIER ──────────────
print("=" * 60)
print("MODEL 1 — REVERSAL CLASSIFIER (with class balancing)")
print("=" * 60)

bottom_models   = {}
top_models      = {}
bottom_encoders = {}
top_encoders    = {}

for group in sorted(group_stocks.keys()):
    print(f"\n── {group} ({len(group_stocks[group])} stocks) ──")

    # Bottom reversal
    model_b, metrics_b, le_b = walk_forward_classify_v2(group, TARGET_BOTTOM)
    if model_b:
        bottom_models[group]   = model_b
        bottom_encoders[group] = le_b
        print(f"  Bottom reversal:")
        for m in metrics_b:
            print(f"    {m['fold']}: prec={m['precision']} | "
                  f"rec={m['recall']} | f1={m['f1']} | "
                  f"events={m['pos_events']}/{m['test_rows']}")

    # Top reversal
    model_t, metrics_t, le_t = walk_forward_classify_v2(group, TARGET_TOP)
    if model_t:
        top_models[group]   = model_t
        top_encoders[group] = le_t
        print(f"  Top reversal:")
        for m in metrics_t:
            print(f"    {m['fold']}: prec={m['precision']} | "
                  f"rec={m['recall']} | f1={m['f1']} | "
                  f"events={m['pos_events']}/{m['test_rows']}")

print(f"\n✅ Model 1 trained for {len(bottom_models)} groups")

MODEL 1 — REVERSAL CLASSIFIER (with class balancing)

── Chemicals (57 stocks) ──
  Bottom reversal:
    2021: prec=0.266 | rec=0.964 | f1=0.417 | events=501/10508
    2022: prec=0.213 | rec=0.96 | f1=0.349 | events=650/10783
    2023: prec=0.209 | rec=0.887 | f1=0.339 | events=478/11136
    2024: prec=0.259 | rec=0.974 | f1=0.409 | events=744/11655
    2025: prec=0.139 | rec=0.987 | f1=0.244 | events=620/12820
  Top reversal:
    2021: prec=0.118 | rec=0.888 | f1=0.208 | events=481/10508
    2022: prec=0.246 | rec=0.827 | f1=0.38 | events=882/10783
    2023: prec=0.113 | rec=0.886 | f1=0.201 | events=395/11136
    2024: prec=0.219 | rec=0.96 | f1=0.356 | events=886/11655
    2025: prec=0.184 | rec=0.978 | f1=0.309 | events=543/12820

── Consumer (120 stocks) ──
  Bottom reversal:
    2021: prec=0.276 | rec=0.964 | f1=0.429 | events=1166/20667
    2022: prec=0.232 | rec=0.995 | f1=0.377 | events=1406/22417
    2023: prec=0.228 | rec=0.989 | f1=0.37 | events=963/24052
    2024: prec=0.2

In [19]:
# ── CELL 9: TRAIN MODEL 2 — TREND CLASSIFIER ─────────────────
print("=" * 60)
print("MODEL 2 — 3-CLASS TREND CLASSIFIER")
print("=" * 60)
print("Classes: Uptrend / Sideways / Downtrend")
print("Random baseline: 33.3%  |  Target: >45%")

trend_models         = {}
trend_encoders       = {}
trend_label_encoders = {}

for group in sorted(group_stocks.keys()):
    print(f"\n── {group} ({len(group_stocks[group])} stocks) ──")

    model_t, metrics_t, le_sym, le_trend = walk_forward_trend(group)

    if model_t:
        trend_models[group]         = model_t
        trend_encoders[group]       = le_sym
        trend_label_encoders[group] = le_trend

        print(f"  Classes: {list(le_trend.classes_)}")
        for m in metrics_t:
            flag = '✅' if m['accuracy'] > 0.45 else '⚠️ '
            print(f"  {flag} {m['fold']}: acc={m['accuracy']} | test={m['test_rows']} rows")

print(f"\n✅ Model 2 trained for {len(trend_models)} groups")

MODEL 2 — 3-CLASS TREND CLASSIFIER
Classes: Uptrend / Sideways / Downtrend
Random baseline: 33.3%  |  Target: >45%

── Chemicals (57 stocks) ──
  Classes: ['Bearish', 'Bullish', 'Neutral', 'Sideways']
  ⚠️  2021: acc=0.384 | test=10508 rows
  ⚠️  2022: acc=0.376 | test=10783 rows
  ⚠️  2023: acc=0.436 | test=11136 rows
  ⚠️  2024: acc=0.389 | test=11655 rows
  ⚠️  2025: acc=0.392 | test=12820 rows

── Consumer (120 stocks) ──
  Classes: ['Bearish', 'Bullish', 'Neutral', 'Sideways']
  ⚠️  2021: acc=0.441 | test=20667 rows
  ⚠️  2022: acc=0.411 | test=22417 rows
  ✅ 2023: acc=0.487 | test=24052 rows
  ⚠️  2024: acc=0.419 | test=25612 rows
  ✅ 2025: acc=0.458 | test=28025 rows

── Financial (91 stocks) ──
  Classes: ['Bearish', 'Bullish', 'Neutral', 'Sideways']
  ⚠️  2021: acc=0.406 | test=15406 rows
  ⚠️  2022: acc=0.39 | test=16214 rows
  ⚠️  2023: acc=0.426 | test=17396 rows
  ⚠️  2024: acc=0.421 | test=18361 rows
  ✅ 2025: acc=0.46 | test=20044 rows

── Healthcare (68 stocks) ──
  Cla

In [26]:
# ── CELL 10: TRAIN MODEL 3 — PRICE FORECAST REGRESSOR ────────
print("=" * 60)
print("MODEL 3 — PRICE FORECAST REGRESSOR")
print("=" * 60)
print("Horizons: 25d / 45d / 180d per group")
print("Metric: Directional accuracy (sign match)")
print("Baseline: 50% (random coin flip)")

# forecast_models[group][horizon]   = model
# forecast_encoders[group][horizon] = label_encoder
forecast_models   = {}
forecast_encoders = {}

for group in sorted(group_stocks.keys()):
    print(f"\n── {group} ({len(group_stocks[group])} stocks) ──")
    forecast_models[group]   = {}
    forecast_encoders[group] = {}

    for horizon, target_col in [
        ('25d',  TARGET_25D),
        ('45d',  TARGET_45D),
        ('180d', TARGET_180D),
    ]:
        model_f, metrics_f, le_f = walk_forward_regress(group, target_col)

        if model_f:
            forecast_models[group][horizon]   = model_f
            forecast_encoders[group][horizon] = le_f

            dir_accs = [m['dir_acc'] for m in metrics_f]
            avg_dir  = round(np.mean(dir_accs), 3) if dir_accs else 0
            flag     = '✅' if avg_dir > 0.55 else '⚠️ '
            print(f"  {flag} {horizon}: avg dir_acc={avg_dir} | folds={len(metrics_f)}")

print(f"\n✅ Model 3 trained for {len(forecast_models)} groups")


# ── CELL 10: TRAIN MODEL 4 — CONTINUATION CLASSIFIER ─────────
print("=" * 60)
print("MODEL 4 — CONTINUATION CLASSIFIER")
print("=" * 60)
print("Bullish Cont: in uptrend + predicted to continue up")
print("Bearish Cont: in downtrend + predicted to continue down")

bullish_cont_models   = {}
bearish_cont_models   = {}
bullish_cont_encoders = {}
bearish_cont_encoders = {}

for group in sorted(group_stocks.keys()):
    print(f"\n── {group} ({len(group_stocks[group])} stocks) ──")

    # Bullish continuation
    model_bc, metrics_bc, le_bc = walk_forward_continuation(
        group, 'bullish_cont'
    )
    if model_bc:
        bullish_cont_models[group]   = model_bc
        bullish_cont_encoders[group] = le_bc
        print(f"  Bullish continuation:")
        for m in metrics_bc:
            print(f"    {m['fold']}: prec={m['precision']} | "
                  f"rec={m['recall']} | f1={m['f1']} | "
                  f"events={m['pos_events']}/{m['test_rows']}")

    # Bearish continuation
    model_dc, metrics_dc, le_dc = walk_forward_continuation(
        group, 'bearish_cont'
    )
    if model_dc:
        bearish_cont_models[group]   = model_dc
        bearish_cont_encoders[group] = le_dc
        print(f"  Bearish continuation:")
        for m in metrics_dc:
            print(f"    {m['fold']}: prec={m['precision']} | "
                  f"rec={m['recall']} | f1={m['f1']} | "
                  f"events={m['pos_events']}/{m['test_rows']}")

print(f"\n✅ Model 4 trained for {len(bullish_cont_models)} groups")


MODEL 3 — PRICE FORECAST REGRESSOR
Horizons: 25d / 45d / 180d per group
Metric: Directional accuracy (sign match)
Baseline: 50% (random coin flip)

── Chemicals (57 stocks) ──
  ⚠️  25d: avg dir_acc=0.498 | folds=5
  ⚠️  45d: avg dir_acc=0.512 | folds=5
  ✅ 180d: avg dir_acc=0.569 | folds=5

── Consumer (120 stocks) ──
  ⚠️  25d: avg dir_acc=0.52 | folds=5
  ⚠️  45d: avg dir_acc=0.541 | folds=5
  ✅ 180d: avg dir_acc=0.592 | folds=5

── Financial (91 stocks) ──
  ⚠️  25d: avg dir_acc=0.532 | folds=5
  ⚠️  45d: avg dir_acc=0.535 | folds=5
  ✅ 180d: avg dir_acc=0.625 | folds=5

── Healthcare (68 stocks) ──
  ⚠️  25d: avg dir_acc=0.535 | folds=5
  ⚠️  45d: avg dir_acc=0.547 | folds=5
  ✅ 180d: avg dir_acc=0.608 | folds=5

── IT (60 stocks) ──
  ⚠️  25d: avg dir_acc=0.52 | folds=5
  ⚠️  45d: avg dir_acc=0.529 | folds=5
  ✅ 180d: avg dir_acc=0.555 | folds=5

── Industrial (356 stocks) ──
  ⚠️  25d: avg dir_acc=0.545 | folds=5
  ✅ 45d: avg dir_acc=0.559 | folds=5
  ✅ 180d: avg dir_acc=0.643 |

In [27]:
# ── CELL 11: SAVE ALL MODELS ──────────────────────────────────

print("Saving models...")

with open('../models/bottom_models.pkl', 'wb') as f:
    pickle.dump(bottom_models, f)
with open('../models/bottom_encoders.pkl', 'wb') as f:
    pickle.dump(bottom_encoders, f)

with open('../models/top_models.pkl', 'wb') as f:
    pickle.dump(top_models, f)
with open('../models/top_encoders.pkl', 'wb') as f:
    pickle.dump(top_encoders, f)

with open('../models/trend_models.pkl', 'wb') as f:
    pickle.dump(trend_models, f)
with open('../models/trend_encoders.pkl', 'wb') as f:
    pickle.dump(trend_encoders, f)
with open('../models/trend_label_encoders.pkl', 'wb') as f:
    pickle.dump(trend_label_encoders, f)

with open('../models/forecast_models.pkl', 'wb') as f:
    pickle.dump(forecast_models, f)
with open('../models/forecast_encoders.pkl', 'wb') as f:
    pickle.dump(forecast_encoders, f)

# ── NEW: CONTINUATION MODELS ──────────────────────────────────
with open('../models/bullish_cont_models.pkl', 'wb') as f:
    pickle.dump(bullish_cont_models, f)
with open('../models/bullish_cont_encoders.pkl', 'wb') as f:
    pickle.dump(bullish_cont_encoders, f)

with open('../models/bearish_cont_models.pkl', 'wb') as f:
    pickle.dump(bearish_cont_models, f)
with open('../models/bearish_cont_encoders.pkl', 'wb') as f:
    pickle.dump(bearish_cont_encoders, f)

# ── GROUP MAPPINGS ────────────────────────────────────────────
with open('../models/symbol_group.pkl', 'wb') as f:
    pickle.dump(symbol_group, f)
with open('../models/group_stocks.pkl', 'wb') as f:
    pickle.dump(dict(group_stocks), f)

# Verify
saved = os.listdir('../models/')
print(f"\n✅ Models saved to /models/")
for fname in sorted(saved):
    size = os.path.getsize(f'../models/{fname}') / 1024
    print(f"  {fname:45} {size:6.1f} KB")

Saving models...

✅ Models saved to /models/
  bearish_cont_encoders.pkl                        8.0 KB
  bearish_cont_models.pkl                       1504.6 KB
  bottom_encoders.pkl                              8.0 KB
  bottom_models.pkl                             1692.5 KB
  bullish_cont_encoders.pkl                        8.0 KB
  bullish_cont_models.pkl                       1460.3 KB
  forecast_encoders.pkl                           22.6 KB
  forecast_models.pkl                           6031.1 KB
  group_stocks.pkl                                 7.6 KB
  symbol_group.pkl                                 9.0 KB
  top_encoders.pkl                                 8.0 KB
  top_models.pkl                                1682.0 KB
  trend_encoders.pkl                               8.0 KB
  trend_label_encoders.pkl                         0.8 KB
  trend_models.pkl                              18712.8 KB


In [29]:
# ── CELL 12: INFERENCE ENGINE ─────────────────────────────────

def get_latest_features(symbol):
    df = all_features[symbol].copy()
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=FEATURE_COLS)
    if len(df) == 0:
        return None
    return df.iloc[[-1]]

def get_symbol_enc(symbol, encoder):
    try:
        return encoder.transform([symbol])[0]
    except:
        return 0

def assign_ml_label(best_setup, forecast_25d):
    if best_setup == 'Momentum':
        return 'Bullish Continual' if forecast_25d >= 0 else 'Bearish Continual'
    elif best_setup == 'Reversal':
        return 'Reversal' if forecast_25d >= 0 else 'Bearish'
    elif best_setup == 'Watching':
        return 'Bullish' if forecast_25d >= 0 else 'No Signal'
    else:
        return 'No Signal'

def run_inference(symbol, best_setup):
    result = {
        'Symbol'             : symbol,
        'Group'              : None,
        'Bottom_Rev_Prob'    : None,
        'Top_Rev_Prob'       : None,
        'Bottom_Rev_Flag'    : False,
        'Top_Rev_Flag'       : False,
        'Bullish_Cont_Prob'  : None,
        'Bearish_Cont_Prob'  : None,
        'Forecast_25d_Pct'   : None,
        'Forecast_45d_Pct'   : None,
        'Forecast_180d_Pct'  : None,
        'Forecast_25d_Price' : None,
        'Forecast_45d_Price' : None,
        'Forecast_180d_Price': None,
        'ML_Prediction'      : 'No Signal',
        'ML_Confidence'      : None,
    }

    group = symbol_group.get(symbol)
    if group is None:
        return result
    result['Group'] = group

    latest = get_latest_features(symbol)
    if latest is None:
        return result

    feat_cols = FEATURE_COLS + ['symbol_enc']

    # ── MODEL 1: REVERSAL ─────────────────────────────────────
    if group in bottom_models:
        sym_enc              = get_symbol_enc(symbol, bottom_encoders[group])
        latest['symbol_enc'] = sym_enc
        X                    = latest[feat_cols]

        bottom_prob = bottom_models[group].predict_proba(X)[0][1]
        result['Bottom_Rev_Prob'] = round(float(bottom_prob) * 100, 1)
        result['Bottom_Rev_Flag'] = bottom_prob >= 0.60

        top_prob = top_models[group].predict_proba(X)[0][1]
        result['Top_Rev_Prob'] = round(float(top_prob) * 100, 1)
        result['Top_Rev_Flag'] = top_prob >= 0.60

    # ── MODEL 4: CONTINUATION ─────────────────────────────────
    if group in bullish_cont_models:
        sym_enc              = get_symbol_enc(symbol, bullish_cont_encoders[group])
        latest['symbol_enc'] = sym_enc
        X                    = latest[feat_cols]

        bc_prob = bullish_cont_models[group].predict_proba(X)[0][1]
        result['Bullish_Cont_Prob'] = round(float(bc_prob) * 100, 1)

    if group in bearish_cont_models:
        sym_enc              = get_symbol_enc(symbol, bearish_cont_encoders[group])
        latest['symbol_enc'] = sym_enc
        X                    = latest[feat_cols]

        dc_prob = bearish_cont_models[group].predict_proba(X)[0][1]
        result['Bearish_Cont_Prob'] = round(float(dc_prob) * 100, 1)

    # ── MODEL 3: FORECAST ─────────────────────────────────────
    if group in forecast_models:
        sym_enc              = get_symbol_enc(symbol, forecast_encoders[group]['25d'])
        latest['symbol_enc'] = sym_enc
        X                    = latest[feat_cols]

        current_price = all_features[symbol]['Close'].iloc[-1]

        for horizon in ['25d', '45d', '180d']:
            if horizon not in forecast_models[group]:
                continue
            pred_return = forecast_models[group][horizon].predict(X)[0]
            pred_price  = current_price * (1 + pred_return)
            result[f'Forecast_{horizon}_Pct']   = round(float(pred_return) * 100, 1)
            result[f'Forecast_{horizon}_Price']  = round(float(pred_price), 2)

    # ── ASSIGN ML LABEL ───────────────────────────────────────
    forecast_25d            = result['Forecast_25d_Pct'] or 0
    result['ML_Prediction'] = assign_ml_label(best_setup, forecast_25d)

    # ── ML CONFIDENCE ─────────────────────────────────────────
    # Each label uses the most appropriate model probability
    label = result['ML_Prediction']

    if label == 'Bullish Continual':
        result['ML_Confidence'] = result['Bullish_Cont_Prob']

    elif label == 'Bearish Continual':
        result['ML_Confidence'] = result['Bearish_Cont_Prob']

    elif label == 'Reversal':
        result['ML_Confidence'] = result['Bottom_Rev_Prob']

    elif label == 'Bearish':
        result['ML_Confidence'] = result['Top_Rev_Prob']

    elif label == 'Bullish':
        # Watching + positive forecast → use bottom reversal prob as proxy
        # continuation model gives 0 since stock not in established uptrend
        result['ML_Confidence'] = result['Bottom_Rev_Prob']

    else:
        result['ML_Confidence'] = None

    return result

# ── TEST ON SAMPLE STOCKS ─────────────────────────────────────
tech_df    = pd.read_csv('../data/technical_report_full.csv')

# Test one from each Best Setup category
momentum_ex = tech_df[tech_df['Best Setup'] == 'Momentum']['Symbol'].iloc[0]
reversal_ex = tech_df[tech_df['Best Setup'] == 'Reversal']['Symbol'].iloc[0]
watching_ex = tech_df[tech_df['Best Setup'] == 'Watching']['Symbol'].iloc[0]

for sym, setup in [
    (momentum_ex, 'Momentum'),
    (reversal_ex, 'Reversal'),
    (watching_ex, 'Watching'),
]:
    r = run_inference(sym, setup)
    print(f"\n{sym} [{setup}]:")
    print(f"  ML Prediction    : {r['ML_Prediction']}")
    print(f"  ML Confidence    : {r['ML_Confidence']}%")
    print(f"  Forecast 25d     : {r['Forecast_25d_Pct']}%")
    print(f"  Bullish Cont Prob: {r['Bullish_Cont_Prob']}%")
    print(f"  Bearish Cont Prob: {r['Bearish_Cont_Prob']}%")
    print(f"  Bottom Rev Prob  : {r['Bottom_Rev_Prob']}%")
    print(f"  Top Rev Prob     : {r['Top_Rev_Prob']}%")


ABB [Momentum]:
  ML Prediction    : Bullish Continual
  ML Confidence    : 75.7%
  Forecast 25d     : 1.9%
  Bullish Cont Prob: 75.7%
  Bearish Cont Prob: 0.0%
  Bottom Rev Prob  : 0.0%
  Top Rev Prob     : 68.1%

360ONE [Reversal]:
  ML Prediction    : Reversal
  ML Confidence    : 72.6%
  Forecast 25d     : 2.5%
  Bullish Cont Prob: 0.0%
  Bearish Cont Prob: 0.1%
  Bottom Rev Prob  : 72.6%
  Top Rev Prob     : 0.0%

20MICRONS [Watching]:
  ML Prediction    : Bullish
  ML Confidence    : 83.5%
  Forecast 25d     : 1.3%
  Bullish Cont Prob: 0.0%
  Bearish Cont Prob: 85.0%
  Bottom Rev Prob  : 83.5%
  Top Rev Prob     : 0.0%


In [30]:
# ── CELL 13: RUN INFERENCE ON ALL STOCKS + MERGE INTO TECH REPORT ──

print("Running inference on all stocks...")
print("=" * 60)

tech_df       = pd.read_csv('../data/technical_report_full.csv')
all_ml_scores = []
failed        = []

for _, row in tech_df.iterrows():
    symbol     = row['Symbol']
    best_setup = row['Best Setup']
    try:
        result = run_inference(symbol, best_setup)
        all_ml_scores.append(result)
    except Exception as e:
        failed.append(symbol)
        print(f"  ⚠️  {symbol}: {e}")

ml_scores_df = pd.DataFrame(all_ml_scores)

# ── MERGE INTO TECH REPORT ────────────────────────────────────
merge_cols = [
    'Symbol',
    'ML_Prediction',
    'ML_Confidence',
    'Forecast_25d_Pct',
    'Forecast_45d_Pct',
    'Forecast_180d_Pct',
    'Forecast_25d_Price',
    'Forecast_45d_Price',
    'Forecast_180d_Price',
    'Bottom_Rev_Prob',
    'Top_Rev_Prob',
    'Bottom_Rev_Flag',
    'Top_Rev_Flag',
    'Bullish_Cont_Prob',
    'Bearish_Cont_Prob',
]

# Drop old ML columns if they exist from a previous run
for col in merge_cols[1:]:
    if col in tech_df.columns:
        tech_df = tech_df.drop(columns=[col])

tech_df = tech_df.merge(
    ml_scores_df[merge_cols], on='Symbol', how='left'
)

# Save enriched tech report
tech_df.to_csv('../data/technical_report_full.csv', index=False)

# Also save standalone ml scores
ml_scores_df.to_csv('../data/ml_scores_full.csv', index=False)

print(f"✅ Inference complete  : {len(ml_scores_df)} stocks")
print(f"   Failed             : {len(failed)}")

print(f"\nML Prediction Distribution:")
print(ml_scores_df['ML_Prediction'].value_counts().to_string())

print(f"\nSample — Bullish Continual (top 5 by confidence):")
bc = ml_scores_df[ml_scores_df['ML_Prediction'] == 'Bullish Continual'].nlargest(
    5, 'ML_Confidence'
)[['Symbol', 'ML_Prediction', 'ML_Confidence', 'Forecast_25d_Pct']]
print(bc.to_string(index=False))

print(f"\nSample — Reversal (top 5 by confidence):")
rv = ml_scores_df[ml_scores_df['ML_Prediction'] == 'Reversal'].nlargest(
    5, 'ML_Confidence'
)[['Symbol', 'ML_Prediction', 'ML_Confidence', 'Forecast_25d_Pct']]
print(rv.to_string(index=False))

print(f"\nSample — Bearish Continual (top 5 by confidence):")
dc = ml_scores_df[ml_scores_df['ML_Prediction'] == 'Bearish Continual'].nlargest(
    5, 'ML_Confidence'
)[['Symbol', 'ML_Prediction', 'ML_Confidence', 'Forecast_25d_Pct']]
print(dc.to_string(index=False))

Running inference on all stocks...
✅ Inference complete  : 752 stocks
   Failed             : 0

ML Prediction Distribution:
ML_Prediction
Reversal             261
Bullish              260
Bullish Continual    112
No Signal             58
Bearish               55
Bearish Continual      6

Sample — Bullish Continual (top 5 by confidence):
   Symbol     ML_Prediction  ML_Confidence  Forecast_25d_Pct
  SATTRIX Bullish Continual           85.7              10.8
ORICONENT Bullish Continual           81.8               4.9
    AGIIL Bullish Continual           81.7               4.7
   GLOBAL Bullish Continual           81.5               4.1
 BAJAJCON Bullish Continual           80.8               5.2

Sample — Reversal (top 5 by confidence):
    Symbol ML_Prediction  ML_Confidence  Forecast_25d_Pct
ASHAPURMIN      Reversal           93.6               8.3
ECOSMOBLTY      Reversal           91.9               8.5
   GENESYS      Reversal           91.6              10.1
    VERTOZ      Reve

In [33]:
# ── CELL 14: GENERATE ML REPORT (SHORT + LONG FORMAT) ─────────

print("Select report format:")
print("  1 = Short Report")
print("  2 = Long Report")
print("  3 = Both")
report_mode = input("Enter 1, 2, or 3: ").strip()
if report_mode not in ['1', '2', '3']:
    report_mode = '2'
    print("Invalid input — defaulting to Long Report")

GENERATE_SHORT = report_mode in ['1', '3']
GENERATE_LONG  = report_mode in ['2', '3']

# ── LOAD ENRICHED TECH REPORT ─────────────────────────────────
tech_df = pd.read_csv('../data/technical_report_full.csv')

# ── CONFIDENCE THRESHOLDS ─────────────────────────────────────
CONF_THRESHOLD = {
    'Bullish Continual' : 40.0,
    'Bullish'           : 50.0,
    'Reversal'          : 50.0,
}

LABEL_DESCRIPTION = {
    'Bullish Continual' : 'Already in uptrend — ML confirms continuation',
    'Bullish'           : 'No clear setup yet — ML sees upside potential',
    'Reversal'          : 'In downtrend — ML sees high bounce probability',
}

# ── TECHNICAL TREND FUNCTION ──────────────────────────────────
def get_tech_trend(row):
    try:
        price   = float(row['Current Price'])
        ema20   = float(row['EMA20'])
        ema50   = float(row['EMA50'])
        ema200  = float(row['EMA200'])
        adx     = float(row['ADX'])
        rsi     = float(row['RSI'])
        macd    = float(row['MACD Hist'])
        vol     = float(row['Vol Ratio'])

        # Direction flags
        full_bull_align  = price > ema20 > ema50 > ema200
        partial_bull     = price > ema50 > ema200
        above_ema200     = price > ema200
        full_bear_align  = price < ema20 and ema20 < ema50 and ema50 < ema200
        partial_bear     = price < ema200 and ema20 < ema50

        # Strength/momentum flags
        strong_trend     = adx > 25
        mild_trend       = adx > 20
        bull_momentum    = rsi > 55 and macd > 0
        bear_momentum    = rsi < 45 and macd < 0
        high_vol         = vol > 1.0

        # ── STRONG UPTREND ────────────────────────────────────
        if full_bull_align and strong_trend and bull_momentum and high_vol:
            return 'Strong Uptrend'

        # ── UPTREND ───────────────────────────────────────────
        elif partial_bull and (mild_trend or rsi > 50 or macd > 0):
            return 'Uptrend'

        # ── STRONG DOWNTREND ──────────────────────────────────
        elif full_bear_align and strong_trend and bear_momentum and high_vol:
            return 'Strong Downtrend'

        # ── DOWNTREND ─────────────────────────────────────────
        elif partial_bear and (mild_trend or macd < 0):
            return 'Downtrend'

        # ── SIDEWAYS ──────────────────────────────────────────
        else:
            return 'Sideways'

    except:
        return 'Unknown'

# Compute tech trend for all stocks
tech_df['Tech_Trend'] = tech_df.apply(get_tech_trend, axis=1)

print("Tech Trend distribution:")
print(tech_df['Tech_Trend'].value_counts().to_string())

# ── HELPER FUNCTIONS ──────────────────────────────────────────
def mcap_str(mcap_cr):
    try:
        v = float(mcap_cr)
        if v >= 100000: return f"Rs{v/100000:.1f}L Cr"
        return f"Rs{v:,.0f}Cr"
    except:
        return "Rs—"

def tier_short(tier):
    return (str(tier)
            .replace('TIER 1 — ', 'T1-')
            .replace('TIER 2 — ', 'T2-')
            .replace('TIER 3 — ', 'T3-')
            .replace('BUY NOW (Momentum)', 'MOM')
            .replace('BUY NOW (Reversal)', 'REV')
            .replace('BREAKOUT IMMINENT', 'BRKOUT')
            .replace('WATCHLIST',         'WATCH')
            .replace('BASE BUILDING',     'BASE')
            .replace('WAITING',           'WAIT'))

def trend_short(trend):
    return {
        'Strong Uptrend'   : 'Str-Up',
        'Uptrend'          : 'Up',
        'Sideways'         : 'Side',
        'Downtrend'        : 'Down',
        'Strong Downtrend' : 'Str-Dn',
        'Unknown'          : '?',
    }.get(trend, trend)

# ── REPORT GENERATOR ──────────────────────────────────────────
def generate_ml_report(is_short):
    today      = datetime.now().strftime('%d %B %Y')
    mode_label = 'SHORT' if is_short else 'LONG'
    lines      = []

    def p(line=''):
        lines.append(str(line))

    p("═" * 82)
    p(f"  ML SIGNALS — WEEKLY REPORT ({mode_label})")
    p(f"  {today}  |  Universe: {len(tech_df)} stocks")
    p("═" * 82)
    p()
    p("  MODEL NOTE:")
    p("  Confidence = ML model probability for that specific label.")
    p("  Bullish Continual ≥40% | Bullish ≥50% | Reversal ≥50%")
    p("  SecRank = rank vs same sector | CapRank = rank vs sector+cap")
    p("  Trend: Str-Up=Strong Uptrend | Up=Uptrend | Side=Sideways")
    p("         Down=Downtrend | Str-Dn=Strong Downtrend")
    p("  Trend uses EMA alignment + ADX + RSI + MACD + Volume")
    p("  Forecast accuracy: 25d ~52% | 45d ~54% | use as hint only")

    # ── SUMMARY ───────────────────────────────────────────────
    p()
    p("─" * 82)
    p("  SUMMARY")
    p("─" * 82)
    for label, threshold in CONF_THRESHOLD.items():
        subset = tech_df[
            (tech_df['ML_Prediction'] == label) &
            (tech_df['ML_Confidence'] >= threshold)
        ]
        avg_conf = subset['ML_Confidence'].mean()
        p(f"  {label:<22} : {len(subset):>3} stocks "
          f"| Avg confidence: {avg_conf:.1f}%")

    # ── SECTIONS ──────────────────────────────────────────────
    for label, threshold in CONF_THRESHOLD.items():

        subset = tech_df[
            (tech_df['ML_Prediction'] == label) &
            (tech_df['ML_Confidence'] >= threshold)
        ].copy()

        if len(subset) == 0:
            continue

        # Sort by confidence descending
        subset = subset.sort_values(
            'ML_Confidence', ascending=False
        ).reset_index(drop=True)

        p()
        p("─" * 82)
        p(f"  {label.upper()}  ({len(subset)} stocks)")
        p(f"  {LABEL_DESCRIPTION[label]}")
        p("─" * 82)

        # Header
        p(f"  {'#':<4} {'Symbol':<14} {'Cap':<16} "
          f"{'SecRnk':>7} {'CapRnk':>7} {'Conf':>6} "
          f"{'25d%':>6} {'45d%':>6}  {'Trend':<8}  Tier")
        p("  " + "─" * 78)

        for i, row in subset.iterrows():
            sym     = str(row['Symbol'])
            cap     = str(row['Cap Category'])
            sec_rnk = float(row.get('Sector Score', 0) or 0)
            cap_rnk = float(row.get('Cap Score',    0) or 0)
            conf    = float(row['ML_Confidence'] or 0)
            f25     = float(row['Forecast_25d_Pct'] or 0)
            f45     = float(row['Forecast_45d_Pct'] or 0)
            t       = tier_short(row['Tier'])
            trend   = trend_short(row.get('Tech_Trend', 'Unknown'))

            p(f"  {i+1:<4} {sym:<14} {cap:<16} "
              f"{sec_rnk:>6.1f}  {cap_rnk:>6.1f}  "
              f"{conf:>5.1f}% {f25:>+5.1f}% {f45:>+5.1f}%  "
              f"{trend:<8}  {t}")

            # Long format: price targets + sector + consolidation
            if not is_short:
                price = float(row['Current Price'])
                p25   = float(row['Forecast_25d_Price'] or 0)
                p45   = float(row['Forecast_45d_Price'] or 0)
                mcap  = mcap_str(row['Market Cap Cr'])
                sect  = str(row['Sector'])[:30]

                p(f"       {sect}  |  MCap {mcap}")
                p(f"       Price: Rs{price:.2f} → "
                  f"25d: Rs{p25:.0f} | 45d: Rs{p45:.0f}")

                # Consolidation info
                if row.get('In Consolidation') == True:
                    days    = int(row['Consol Days'])
                    pct_brk = float(row['Pct to Breakout'] or 0)
                    p(f"       Base: {days}d | "
                      f"{pct_brk:+.1f}% to breakout")

                # 180d only for Bullish Continual
                if label == 'Bullish Continual':
                    f180 = float(row.get('Forecast_180d_Pct') or 0)
                    p180 = float(row.get('Forecast_180d_Price') or 0)
                    p(f"       180d: {f180:+.1f}% → Rs{p180:.0f}")

                p()

    # ── FOOTER ────────────────────────────────────────────────
    p("═" * 82)
    p("  Bullish Continual = uptrend confirmed, ML sees continuation")
    p("  Bullish           = potential upside, no confirmed setup yet")
    p("  Reversal          = oversold, ML sees high bounce probability")
    p("  SecRnk/CapRnk out of 10 — higher = stronger relative rank")
    p("  Trend = EMA alignment + ADX strength + RSI + MACD + Volume")
    p("═" * 82)

    return lines

# ── RUN & SAVE ────────────────────────────────────────────────
today_str  = datetime.now().strftime('%Y-%m-%d')
weekly_dir = '../data/weekly_reports'
os.makedirs(weekly_dir, exist_ok=True)

if GENERATE_SHORT:
    short_lines = generate_ml_report(is_short=True)
    short_path  = f'{weekly_dir}/ml_report_short_{today_str}.txt'
    with open(short_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(short_lines))
    print(f"✅ Short ML report saved: {short_path}")
    print('\n'.join(short_lines[:80]))

if GENERATE_LONG:
    long_lines = generate_ml_report(is_short=False)
    long_path  = f'{weekly_dir}/ml_report_long_{today_str}.txt'
    with open(long_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(long_lines))
    print(f"✅ Long ML report saved: {long_path}")

Select report format:
  1 = Short Report
  2 = Long Report
  3 = Both


Enter 1, 2, or 3:  2


Tech Trend distribution:
Tech_Trend
Downtrend           449
Sideways            128
Strong Downtrend    104
Uptrend              66
Strong Uptrend        5
✅ Long ML report saved: ../data/weekly_reports/ml_report_long_2026-04-12.txt
